# 第 8 章 ナイーブベイズ

学習ループのない学習器です。単語を数え上げ、ラプラス平滑化を掛け、対数空間で確率を合成します。

対応する記事: [第 8 章 ナイーブベイズ（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch08.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch08_naive_bayes import *

## データセット

スパム 3 通、通常メール 5 通の小さなデータです。**学習は数え上げの 1 パスで終わります。**

In [2]:
documents = [
    "lottery sale", "lottery winning", "winning lottery sale", "sale today",
    "meeting tomorrow", "project meeting", "lunch meeting today", "project deadline",
]
labels = [1, 1, 1, 0, 0, 0, 0, 0]

model = train(documents, labels)
print(f"スパム {model.spam_documents} 通 / 通常 {model.ham_documents} 通")
print(f"事前確率 {prior_spam_probability(model):.4f}")

スパム 3 通 / 通常 5 通
事前確率 0.3750


## ラプラス平滑化

`lottery` はスパム 3 件・通常 0 件です。**平滑化がないと確率 1.0 になり、他のどんな単語が来ても覆せません。** すべてのカウントに 1 を足すことで 0.8 に収まります。

未知語はちょうど 0.5 になり、**判定に寄与しません。**

In [3]:
print(f"{'単語':<10} {'スパム':>6} {'通常':>6} {'確率':>8}")
for word in ["lottery", "winning", "sale", "today", "meeting", "unseen"]:
    print(f"{word:<10} {model.spam_word_counts[word]:>6} {model.ham_word_counts[word]:>6} "
          f"{word_spam_probability(model, word):>8.4f}")

単語            スパム     通常       確率
lottery         3      0   0.8000
winning         2      0   0.7500
sale            2      1   0.6000
today           0      2   0.2500
meeting         0      3   0.2000
unseen          0      0   0.5000


## 証拠が積み重なる

スパム語が重なるほど確率が上がります。これが掛け算（実装上は対数の足し算）の効果です。

**空の文書は事前確率そのもの** を返します。単語による更新が 1 つも起きないためです。

In [4]:
for document in ["", "project deadline", "sale today", "lottery",
                 "lottery winning", "lottery winning sale"]:
    probability = predict_probability(model, document)
    bar = "#" * int(probability * 40)
    print(f"{document or '(空)':<24} {probability:.4f} {bar}")

(空)                      0.3750 ###############
project deadline         0.0909 ###
sale today               0.2308 #########
lottery                  0.7059 ############################
lottery winning          0.8780 ###################################
lottery winning sale     0.9153 ####################################


## 未知語は予測を変えない

平滑化により未知語の確率は 0.5 なので、含めても含めなくても結果は同じです。

In [5]:
print(f"lottery              {predict_probability(model, 'lottery'):.6f}")
print(f"lottery zzzz qqqq    {predict_probability(model, 'lottery zzzz qqqq'):.6f}")
print(f"正解率 {accuracy(model, documents, labels):.2f}")

lottery              0.705882
lottery zzzz qqqq    0.705882
正解率 1.00


## 試してみる: 学習データを足す

新しいメールを 1 通足すと、単語の確率がどう動くでしょうか。**学習ループがないので、足して数え直すだけです。**

In [6]:
extended = train(documents + ["meeting lottery"], labels + [0])

print(f"{'単語':<10} {'元':>8} {'追加後':>8}")
for word in ["lottery", "meeting"]:
    print(f"{word:<10} {word_spam_probability(model, word):>8.4f} "
          f"{word_spam_probability(extended, word):>8.4f}")

単語                元      追加後
lottery      0.8000   0.6667
meeting      0.2000   0.1667
